# Drone YOLO Training — Google Colab

Train a drone detector in a few steps. **Upload your dataset folder** (as a zip) and run the cells.

---

## Required folder structure

Your dataset zip must contain this layout:

```
drone_dataset/
├── images/
│   ├── train/    ← training images (.jpg, .png)
│   ├── val/      ← validation images
│   └── test/     ← test images (optional)
└── labels/
    ├── train/    ← one .txt per image (same filename)
    ├── val/
    └── test/
```

**Label format** (YOLO): one line per box, normalized to [0, 1]:
```
0 x_center y_center width height
```
Example: `0 0.45 0.19 0.04 0.05`

Class `0` = drone. Empty `.txt` files are allowed (no drones in frame).

## 1. Setup — GPU & Dependencies

In [ ]:
# Enable GPU: Runtime → Change runtime type → T4 GPU
!nvidia-smi

In [ ]:
!pip install -q ultralytics

## 2. Import your dataset

**Option A:** Upload a zip file (recommended)
- Zip your `drone_dataset` folder so the zip contains `drone_dataset/images/` and `drone_dataset/labels/`
- Run the cell below and click **Choose Files** to upload

In [ ]:
from google.colab import files
import zipfile
import shutil
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    print("No file uploaded. Upload your dataset zip and re-run this cell.")
else:
    zip_path = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")

    # Find dataset root: prefer drone_dataset/, else use folder with images/ and labels/
    content = Path("/content")
    if (content / "drone_dataset" / "images").exists():
        DATASET_DIR = content / "drone_dataset"
    elif (content / "images").exists() and (content / "labels").exists():
        DATASET_DIR = content / "drone_dataset"
        DATASET_DIR.mkdir(exist_ok=True)
        for d in ["images", "labels"]:
            shutil.move(str(content / d), str(DATASET_DIR / d))
        print("Moved images/ and labels/ into drone_dataset/")
    else:
        # Use first dir that has both images and labels
        for d in content.iterdir():
            if d.is_dir() and (d / "images").exists() and (d / "labels").exists():
                DATASET_DIR = d
                break
        else:
            DATASET_DIR = content / "drone_dataset"
            print("Could not find images/ and labels/. Check your zip structure.")

    print(f"Dataset at: {DATASET_DIR}")
    !ls -la {DATASET_DIR}
    !ls {DATASET_DIR}/images/ 2>/dev/null || echo "Expected: images/train, images/val, images/test"
    !ls {DATASET_DIR}/labels/ 2>/dev/null || echo "Expected: labels/train, labels/val, labels/test"

**Option B:** Use a folder from Google Drive
- Put your dataset in Drive, e.g. `My Drive/drone_dataset/`
- Update `DRIVE_DATASET_PATH` below to match

In [ ]:
# Uncomment to use Google Drive instead of upload
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_DATASET_PATH = "/content/drive/MyDrive/drone_dataset"  # ← change to your path
# DATASET_DIR = Path(DRIVE_DATASET_PATH)
# !ls {DATASET_DIR}

## 3. Create dataset config

In [ ]:
try:
    DATASET_DIR  # Set by upload or Drive cell
except NameError:
    DATASET_DIR = Path("/content/drone_dataset")

CONFIG_PATH = Path("/content/drone_dataset.yaml")

config = f"""
path: {DATASET_DIR}
train: images/train
val: images/val
test: images/test

names:
  0: drone
"""

CONFIG_PATH.write_text(config.strip())
print(f"Config saved to {CONFIG_PATH}")
print(CONFIG_PATH.read_text())

## 4. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data="/content/drone_dataset.yaml",
    epochs=40,
    imgsz=960,
    batch=16,
    patience=12,
    close_mosaic=10,
    pretrained=True,
    plots=True,
    workers=2,
)

print(f"\nDone! Best weights: {results.save_dir}/weights/best.pt")

**If you run out of memory:** reduce `batch` to `8` or `imgsz` to `832`.

## 5. Download trained model

In [ ]:
from google.colab import files
from pathlib import Path

best_pt = Path("/content/runs/detect/train/weights/best.pt")
if best_pt.exists():
    files.download(str(best_pt))
    print("Downloaded best.pt")
else:
    print(f"Not found: {best_pt}. Run the training cell first.")